# Channel Ablation — Corrected Channel Order

1. the images are passed to `model.predict()` exactly as `cv2.imread`
returns them (BGR). Ultralytics flips numpy arrays BGR->RGB internally, once,
which is the same thing it does in training/validation. The old
`cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` caused a second flip, so the network
saw B in the slot it was trained to hold R/Y/L.

With a BGR array the indices are:

| index | RGB model | YGB model | LGB model |
|---|---|---|---|
| 0 (Channel 1) | B | B | B |
| 1 (Channel 2) | G | G | G |
| 2 (Channel 3) | R | Y | L |

2. **Per-object evaluation, not per-image-per-class.** Ground truth is
   loaded as real boxes. Each predicted box is matched to at most one GT
   box, and each GT box is matched to at most one prediction (greedy,
   sorted by confidence, IoU >= 0.5). A GT object with no match scores 0.
   This is what fixed the earlier ~3000-objects-became-998-samples issue,
   and the double-counting risk of matching every GT box independently.
3. **`PREDICT_CONF = 0.001`, one value, used everywhere** — no detections
   filtered out.
4. **Each color space runs on its own full test set independently.** No
   requirement that an image exist in all three folders.

   **Before running:** set `DATASET` below to `'SIMD'` or `'SSL'`, and set
`base_path` / `pt_paths` / `val_img_paths` for that dataset.


In [8]:
import os
import cv2
import numpy as np
from tqdm import tqdm
import csv
from ultralytics import YOLO


In [9]:
base_path = r"C:\Users\DELL\Desktop\c"

pt_paths = {
    'RGB': os.path.join(base_path, r'runs\detect\runs\SIMD_RGB_YOLOv12n_weighted\weights\best.pt'),
    'YGB': os.path.join(base_path, r'runs\detect\runs\SIMD_YGB_YOLOv12n_weighted\weights\best.pt'),
    'LGB': os.path.join(base_path, r'runs\detect\runs\SIMD_LGB_YOLOv12n_weighted\weights\best.pt'),
}
yolo_models = {name: YOLO(path) for name, path in pt_paths.items()}

val_img_paths = {
    'RGB': os.path.join(base_path, r"SIMD_RGB\test\images"),
    'YGB': os.path.join(base_path, r"SIMD_YGB\test\images"),
    'LGB': os.path.join(base_path, r"SIMD_LGB\test\images"),
}

NUM_CLASSES = 15
CLASS_IDS = list(range(NUM_CLASSES))
CLASS_NAMES = {
    0: 'car', 1: 'truck', 2: 'van', 3: 'longvehicle', 4: 'bus',
    5: 'airliner', 6: 'propeller', 7: 'trainer', 8: 'chartered',
    9: 'fighter', 10: 'other', 11: 'stairtruck', 12: 'pushbacktruck',
    13: 'helicopter', 14: 'boat',
}

PREDICT_CONF = 0.001   # defined ONCE, used everywhere

# 'zero' = set the removed channel to 0 (what you had before)
# 'mean' = set the removed channel to its own per-image mean (less
#          off-distribution: removes spatial detail but keeps overall level)
FILL_MODE = 'zero'

# Names for a BGR array: index 0 = B, 1 = G, 2 = R / Y / L
channel_names = {
    'RGB': {1: 'B', 2: 'G', 3: 'R'},
    'YGB': {1: 'B', 2: 'G', 3: 'Y'},
    'LGB': {1: 'B', 2: 'G', 3: 'L'},
}

mask_configs = [
    {'name': 'Full (Baseline)', 'channel': None, 'type': 'none'},
    {'name': 'Mask Channel 1', 'channel': 1, 'type': 'zero_ch'},
    {'name': 'Mask Channel 2', 'channel': 2, 'type': 'zero_ch'},
    {'name': 'Mask Channel 3', 'channel': 3, 'type': 'zero_ch'},
    {'name': 'Keep Only Channel 1', 'channel': 1, 'type': 'keep_only'},
    {'name': 'Keep Only Channel 2', 'channel': 2, 'type': 'keep_only'},
    {'name': 'Keep Only Channel 3', 'channel': 3, 'type': 'keep_only'},
]


In [10]:
def apply_channel_mask(img, channel, mask_type, fill_mode='zero'):
    """img is a BGR uint8 array straight from cv2.imread. channel is 1-indexed."""
    out = img.copy()

    def fill(c):
        out[:, :, c] = 0 if fill_mode == 'zero' else int(round(img[:, :, c].mean()))

    if mask_type == 'zero_ch':
        fill(channel - 1)
    elif mask_type == 'keep_only':
        for c in range(3):
            if c != channel - 1:
                fill(c)
    return out

def label_path_for_image(img_path):
    base, _ = os.path.splitext(img_path)
    base = base.replace(os.sep + 'images' + os.sep, os.sep + 'labels' + os.sep)
    return base + '.txt'


def get_gt_boxes(lbl_path, img_w, img_h, num_classes):
    """Return list of (cls, (x1,y1,x2,y2)) in pixel coords."""
    boxes = []
    if not os.path.exists(lbl_path):
        return boxes
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            c = int(float(parts[0]))
            if not (0 <= c < num_classes):
                continue
            xc, yc, w, h = map(float, parts[1:5])
            x1 = (xc - w / 2) * img_w
            y1 = (yc - h / 2) * img_h
            x2 = (xc + w / 2) * img_w
            y2 = (yc + h / 2) * img_h
            boxes.append((c, (x1, y1, x2, y2)))
    return boxes

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


In [11]:
all_images = [f for f in os.listdir(val_img_paths['RGB'])
              if f.endswith(('.jpg', '.jpeg', '.png'))]

img_list = [f for f in all_images
            if all(os.path.exists(os.path.join(val_img_paths[m], f)) for m in ['RGB', 'YGB', 'LGB'])
            and os.path.exists(label_path_for_image(os.path.join(val_img_paths['RGB'], f)))]

print(f"Using {len(img_list)} images")


Using 500 images


## Sanity check (run before the main loop)

Predicting from the file path (what validation does) must give the same
confidence as predicting from `cv2.imread(...)` with no color conversion.
If this assertion fails, the pipeline is still not equal to validation.


In [12]:
for model_name in ['RGB', 'YGB', 'LGB']:
    m = yolo_models[model_name]
    for n in img_list[:5]:
        p = os.path.join(val_img_paths[model_name], n)
        a = m.predict(p, conf=PREDICT_CONF, verbose=False)[0].boxes.conf.max().item()
        b = m.predict(cv2.imread(p), conf=PREDICT_CONF, verbose=False)[0].boxes.conf.max().item()
        assert abs(a - b) < 1e-6, f"{model_name} {n}: path={a:.4f} vs ndarray={b:.4f}"
print("OK: ndarray input matches file-path input for all three models")


OK: ndarray input matches file-path input for all three models


In [19]:
IOU_THR = 0.5

results = {m: {cfg['name']: {c: [] for c in CLASS_IDS} for cfg in mask_configs}
           for m in ['RGB', 'YGB', 'LGB']}
tp      = {m: {cfg['name']: {c: 0  for c in CLASS_IDS} for cfg in mask_configs}
           for m in ['RGB', 'YGB', 'LGB']}
n_gt    = {m: {cfg['name']: {c: 0  for c in CLASS_IDS} for cfg in mask_configs}
           for m in ['RGB', 'YGB', 'LGB']}

for img_name in tqdm(img_list, desc="Processing images"):
    # GT boxes from RGB labels (identical across color spaces)
    rgb_path = os.path.join(val_img_paths['RGB'], img_name)
    _rgb = cv2.imread(rgb_path)
    if _rgb is None:
        continue
    H, W = _rgb.shape[:2]
    gt_boxes = get_gt_boxes(label_path_for_image(rgb_path), W, H, NUM_CLASSES)
    if not gt_boxes:
        continue

    for model_name in ['RGB', 'YGB', 'LGB']:
        img = cv2.imread(os.path.join(val_img_paths[model_name], img_name))  # BGR
        if img is None:
            continue
        yolo_model = yolo_models[model_name]

        for cfg in mask_configs:
            img_in = img if cfg['type'] == 'none' else apply_channel_mask(
                img, cfg['channel'], cfg['type'], FILL_MODE)

            res = yolo_model.predict(img_in, imgsz=640, conf=PREDICT_CONF, verbose=False)
            boxes = res[0].boxes
            preds = []
            if boxes is not None and len(boxes) > 0:
                cls_arr  = boxes.cls.cpu().numpy()
                conf_arr = boxes.conf.cpu().numpy()
                xyxy_arr = boxes.xyxy.cpu().numpy()
                for c, p, bb in zip(cls_arr, conf_arr, xyxy_arr):
                    preds.append((int(c), float(p), tuple(map(float, bb))))

            for gt_c, gt_box in gt_boxes:
                cands = [p for (pc, p, pb) in preds
                         if pc == gt_c and iou(gt_box, pb) >= IOU_THR]
                score = max(cands) if cands else 0.0
                results[model_name][cfg['name']][gt_c].append(score)
                n_gt[model_name][cfg['name']][gt_c] += 1
                if cands:
                    tp[model_name][cfg['name']][gt_c] += 1

Processing images: 100%|█████████████████████████████████████████████████████████████| 500/500 [03:06<00:00,  2.69it/s]


In [22]:
summary_rows = []

for model_name in ['RGB', 'YGB', 'LGB']:
    print(f"\n{model_name} MODEL  (fill mode: {FILL_MODE}, IoU>={IOU_THR})")

    base_all = [v for c in CLASS_IDS for v in results[model_name]['Full (Baseline)'][c]]
    base_mean = float(np.mean(base_all)) if base_all else 0.0

    for cfg in mask_configs:
        if cfg['type'] == 'none':
            label = 'Full (Baseline)'
        else:
            ch = channel_names[model_name][cfg['channel']]
            label = f"Mask {ch}" if cfg['type'] == 'zero_ch' else f"Only {ch}"

        print(f"  {label}")
        print(f"    {'Class':<16}{'N':<8}{'MeanConf':<12}{'Recall':<10}{'Std':<10}")

        for c in CLASS_IDS:
            vals = results[model_name][cfg['name']][c]
            if not vals:
                continue
            n = n_gt[model_name][cfg['name']][c]
            m = float(np.mean(vals))
            s = float(np.std(vals))
            r = tp[model_name][cfg['name']][c] / n if n else 0.0
            cname = CLASS_NAMES[c]
            print(f"    {cname:<16}{n:<8}{m:<12.4f}{r:<10.4f}{s:<10.4f}")
            summary_rows.append([model_name, label, c, cname, n,
                                 f"{m:.4f}", f"{r:.4f}", f"{s:.4f}", ''])

        all_vals = [v for c in CLASS_IDS for v in results[model_name][cfg['name']][c]]
        tot_tp   = sum(tp[model_name][cfg['name']][c]   for c in CLASS_IDS)
        tot_n    = sum(n_gt[model_name][cfg['name']][c] for c in CLASS_IDS)
        if all_vals:
            m_all = float(np.mean(all_vals))
            s_all = float(np.std(all_vals))
            r_all = tot_tp / tot_n if tot_n else 0.0
            rel   = m_all / base_mean if base_mean else 0.0
            print(f"    {'ALL':<16}{tot_n:<8}{m_all:<12.4f}{r_all:<10.4f}{s_all:<10.4f}"
                  f"  ({rel*100:.1f}% of own baseline)")
            summary_rows.append([model_name, label, 'ALL', 'ALL', tot_n,
                                 f"{m_all:.4f}", f"{r_all:.4f}", f"{s_all:.4f}", f"{rel:.4f}"])


RGB MODEL  (fill mode: zero, IoU>=0.5)
  Full (Baseline)
    Class           N       MeanConf    Recall    Std       
    car             2183    0.6939      0.9876    0.2014    
    truck           236     0.5371      0.8602    0.3292    
    van             520     0.6071      0.8808    0.2887    
    longvehicle     112     0.5650      0.9375    0.3170    
    bus             185     0.6737      0.9622    0.2385    
    airliner        96      0.8435      0.9896    0.1702    
    propeller       16      0.7211      1.0000    0.1993    
    trainer         64      0.7278      0.9844    0.1719    
    chartered       76      0.7533      0.9868    0.1945    
    fighter         5       0.7713      1.0000    0.1523    
    other           61      0.1350      0.6557    0.1800    
    stairtruck      34      0.2216      0.8824    0.2384    
    pushbacktruck   22      0.0638      0.9091    0.0961    
    helicopter      4       0.1004      1.0000    0.0765    
    boat            1014   